In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Website Smoke Tests\n",
        "\n",
        "A quick set of checks for the static site to confirm the homepage and key pages load correctly and local links/assets exist."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Section 1: Import Testing Libraries\n",
        "import os\n",
        "import re\n",
        "from html.parser import HTMLParser\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Section 2: Configure Target Site URL\n",
        "ROOT = os.path.abspath('.')\n",
        "HTML_FILES = []\n",
        "for dirpath, dirnames, filenames in os.walk(ROOT):\n",
        "    for filename in filenames:\n",
        "        if filename.lower().endswith('.html'):\n",
        "            HTML_FILES.append(os.path.join(dirpath, filename))\n",
        "\n",
        "print(f'Found {len(HTML_FILES)} HTML files to inspect.')\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Section 3: Send HTTP Requests (local file check instead of network)\n",
        "class SimpleLinkParser(HTMLParser):\n",
        "    def __init__(self):\n",
        "        super().__init__()\n",
        "        self.links = []\n",
        "\n",
        "    def handle_starttag(self, tag, attrs):\n",
        "        attrs = dict(attrs)\n",
        "        if tag == 'a' and 'href' in attrs:\n",
        "            self.links.append(attrs['href'])\n",
        "        elif tag == 'img' and 'src' in attrs:\n",
        "            self.links.append(attrs['src'])\n",
        "        elif tag == 'script' and 'src' in attrs:\n",
        "            self.links.append(attrs['src'])\n",
        "        elif tag == 'link' and 'href' in attrs:\n",
        "            self.links.append(attrs['href'])\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Section 4: Validate Response Status (local existence and parse checks)\n",
        "errors = []\n",
        "summary = {'html': 0, 'local_links_checked': 0, 'broken_links': 0}\n",
        "for html_file in sorted(HTML_FILES):\n",
        "    summary['html'] += 1\n",
        "    with open(html_file, 'r', encoding='utf-8', errors='ignore') as f:\n",
        "        content = f.read()\n",
        "    parser = SimpleLinkParser()\n",
        "    parser.feed(content)\n",
        "    for ref in parser.links:\n",
        "        if not ref or re.match(r'^(https?:|mailto:|tel:|javascript:|#|//)', ref):\n",
        "            continue\n",
        "        summary['local_links_checked'] += 1\n",
        "        ref_path = ref.split('#')[0].split('?')[0]\n",
        "        target = os.path.normpath(os.path.join(os.path.dirname(html_file), ref_path))\n",
        "        if os.path.commonpath([ROOT, target]) != ROOT:\n",
        "            continue\n",
        "        if os.path.isdir(target):\n",
        "            continue\n",
        "        if not os.path.exists(target):\n",
        "            summary['broken_links'] += 1\n",
        "            errors.append((html_file, ref, target))\n",
        "\n",
        "print('Validation complete.')\n",
        "print(summary)\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Section 5: Check Page Content\n",
        "for html_file in HTML_FILES:\n",
        "    with open(html_file, 'r', encoding='utf-8', errors='ignore') as f:\n",
        "        text = f.read()\n",
        "    if '<title>' not in text.lower() or '</title>' not in text.lower():\n",
        "        errors.append((html_file, '<title> tag missing', html_file))\n",
        "\n",
        "print(f'Checked title presence on {len(HTML_FILES)} pages.')\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Section 6: Report Test Results\n",
        "if errors:\n",
        "    print('FAIL: Found issues in the site test.')\n",
        "    for issue in errors[:50]:\n",
        "        print('-', issue)\n",
        "    if len(errors) > 50:\n",
        "        print('...and', len(errors) - 50, 'more errors.')\n",
        "else:\n",
        "    print('PASS: No broken local references or missing titles found.')\n"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.x"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}
